### Enviornment Setup

In [1]:
# ==========================================
# CELL 1: SETUP & CREDENTIALS
# ==========================================
!pip install -q --upgrade pip
!pip install -q PyPDF2 pdfplumber pypdf google-generativeai supabase python-dotenv pandas

import os
import re
import json
import time
import pandas as pd
import pdfplumber
import google.generativeai as genai
from supabase import create_client
from kaggle_secrets import UserSecretsClient
from google.api_core import exceptions
from IPython.display import FileLink, display
import shutil

# 1. Fetch Secrets
user_secrets = UserSecretsClient()
GEMINI_API_KEY = user_secrets.get_secret("GEMINI_API_KEY")
SUPABASE_URL = user_secrets.get_secret("SUPABASE_URL")
SUPABASE_SERVICE_ROLE_KEY = user_secrets.get_secret("SUPABASE_SERVICE_ROLE_KEY")

# 2. Configure Clients
genai.configure(api_key=GEMINI_API_KEY)
supabase = create_client(SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY)
USER_UUID = "bcc0f3c6-3600-4daf-acfb-1fe093dc1773" # Your validated Auth ID

# 3. Setup Directories
working_dirs = ['/kaggle/working/cv_uploads', '/kaggle/working/cv_exports']
for directory in working_dirs:
    os.makedirs(directory, exist_ok=True)

print("✅ Dependencies installed, secrets loaded, and directories created!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 22.1 MB/s eta 0:00:0000:0100:01


/usr/local/lib/python3.12/dist-packages/wrapt/importer.py:223: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  self.__wrapped__.exec_module(module)


✅ Dependencies installed, secrets loaded, and directories created!


In [2]:
# ==========================================
# CELL 2: CLEAN DATABASE
# ==========================================
print("🗑️ Preparing to clear existing database records...")

try:
    # Because of ON DELETE CASCADE, wiping candidates clears everything cleanly
    delete_response = supabase.table("candidates").delete().neq("id", "00000000-0000-0000-0000-000000000000").execute()
    print("✅ Database successfully wiped! Ready for a fresh batch of CVs.")
except Exception as e:
    print(f"❌ Error clearing database: {e}")

🗑️ Preparing to clear existing database records...
✅ Database successfully wiped! Ready for a fresh batch of CVs.


In [3]:
# ==========================================
# CELL 3: THE BATCH PROCESSING PIPELINE 
# ==========================================

# --- NORMALIZATION HELPERS ---
def extract_float(value):
    if value is None or str(value).strip() == "": return None
    match = re.search(r'\d+(\.\d+)?', str(value))
    return float(match.group()) if match else None

def extract_int(value):
    if value is None or str(value).strip() == "": return None
    match = re.search(r'\d+', str(value))
    return int(match.group()) if match else None

def clean_string(value):
    if not value or str(value).lower() in ["null", "none", "unknown", "n/a", ""]: return None
    return str(value).strip()

def validate_enum(value, valid_options, default="Unknown"):
    cleaned = clean_string(value)
    if not cleaned: return default
    # Match case-insensitively, but return exact casing needed by DB
    for option in valid_options:
        if cleaned.lower() == option.lower(): return option
    return default

UNIVERSITY_MAP = {
    "nust": "National University of Sciences and Technology",
    "lums": "Lahore University of Management Sciences",
    "giki": "Ghulam Ishaq Khan Institute",
    "fast": "National University of Computer and Emerging Sciences",
    "pu": "University of the Punjab",
    "uet": "University of Engineering and Technology",
    "qau": "Quaid-i-Azam University",
    "comsats": "COMSATS University"
}

def normalize_university(raw_name):
    if not raw_name: return None
    cleaned_name = raw_name.lower().strip()
    if cleaned_name in UNIVERSITY_MAP: return UNIVERSITY_MAP[cleaned_name]
    for key, official_name in UNIVERSITY_MAP.items():
        if key in cleaned_name.split() or f"({key})" in cleaned_name: return official_name
    return raw_name.title().strip()

# --- SYSTEM PROMPT ---
SYSTEM_PROMPT = """
You are an expert HR AI for the 'Talash' academic recruitment system. Extract CV info into STRICT JSON.
CRITICAL RULES:
1. `grade_metric`: Must specify if a grade is a 'GPA', 'Percentage', or 'CGPA'.
2. `is_sse_hssc`: Set to true ONLY if the education is Matric, O-Levels, FSc, HSSC, or SSC.
3. `qs_ranking` / `the_ranking`: If you know the global QS/THE rank of the university, provide it as an integer. Otherwise null.
4. `output_type` ENUM: Must be EXACTLY one of: 'Journal', 'Conference', 'Book', 'Patent', or 'Unknown'.
5. `degree_level` ENUM: Must be EXACTLY one of: 'BS', 'MS', 'PhD', or 'Unknown'.
6. `status` ENUM: Must be EXACTLY one of: 'Completed', 'Ongoing', or 'Unknown'.
7. `skill_category` ENUM: Must be EXACTLY one of: 'Technical', 'Research', 'Soft', 'Tool', or 'Unknown'.

Ensure EXACT JSON structure:
{
  "personal_info": {"full_name": "string", "email": "string", "phone_number": "string", "research_summary": "string", "dob": "string", "marital_status": "string", "fathers_name": "string"},
  "education": [{"degree_name": "string", "specialization": "string", "institution_name": "string", "grade_value": "number", "grade_metric": "string", "passing_year": "number", "is_sse_hssc": "boolean", "qs_ranking": "number", "the_ranking": "number"}],
  "experience": [{"job_title": "string", "organization": "string", "location": "string", "start_date": "string", "end_date": "string", "is_current": "boolean"}],
  "certifications": [{"qualification_name": "string", "institution_name": "string", "passing_year": "number"}],
  "awards": [{"award_type": "string", "detail": "string"}],
  "references": [{"reference_name": "string", "designation": "string", "address": "string", "phone": "string", "email": "string"}],
  "research_outputs": [{"title": "string", "venue_name": "string", "output_type": "Journal", "publication_year": "number", "impact_factor": "number", "author_names": "string", "research_topics": ["topic1"]}],
  "supervision": [{"student_name": "string", "degree_level": "BS", "status": "Completed"}],
  "skills": [{"skill_name": "string", "skill_category": "Technical"}]
}
Return ONLY valid JSON.
"""

# --- DATABASE LOGIC ---
def save_to_database(parsed_json, source_id):
    try:
        # 1. Candidate
        p_info = parsed_json.get("personal_info", {})
        metadata = {
            "dob": clean_string(p_info.get("dob")),
            "marital_status": clean_string(p_info.get("marital_status")),
            "fathers_name": clean_string(p_info.get("fathers_name"))
        }
        cand_res = supabase.table("candidates").insert({
            "created_by": USER_UUID,
            "full_name": clean_string(p_info.get("full_name")) or "Unknown Candidate",
            "email": clean_string(p_info.get("email")),
            "phone_number": clean_string(p_info.get("phone_number")),
            "research_summary": clean_string(p_info.get("research_summary")),
            "metadata": metadata,
            "cv_file_path": source_id,
            "processing_status": "completed"
        }).execute()
        cid = cand_res.data[0]['id']

        # 2. Education
        edu = parsed_json.get("education", [])
        if edu:
            supabase.table("education").insert([{
                "candidate_id": cid, 
                "degree_name": clean_string(e.get("degree_name")), 
                "specialization": clean_string(e.get("specialization")),
                "institution_name": normalize_university(e.get("institution_name")), 
                "grade_value": extract_float(e.get("grade_value")),
                "grade_metric": clean_string(e.get("grade_metric")),
                "passing_year": extract_int(e.get("passing_year")),
                "is_sse_hssc": bool(e.get("is_sse_hssc")),
                "qs_ranking": extract_int(e.get("qs_ranking")),
                "the_ranking": extract_int(e.get("the_ranking"))
            } for e in edu]).execute() 

        # 3. Experience
        exp = parsed_json.get("experience", [])
        if exp:
            supabase.table("experience").insert([{
                "candidate_id": cid, 
                "job_title": clean_string(ex.get("job_title")), 
                "organization": clean_string(ex.get("organization")),
                "location": clean_string(ex.get("location")),
                "start_date": clean_string(ex.get("start_date")),
                "end_date": clean_string(ex.get("end_date")),
                "is_current": bool(ex.get("is_current"))
            } for ex in exp]).execute()

        # 4. Certifications
        certs = parsed_json.get("certifications", [])
        if certs:
            supabase.table("certifications").insert([{
                "candidate_id": cid,
                "qualification_name": clean_string(c.get("qualification_name")),
                "institution_name": clean_string(c.get("institution_name")),
                "passing_year": extract_int(c.get("passing_year"))
            } for c in certs]).execute()

        # 5. Awards
        awards = parsed_json.get("awards", [])
        if awards:
            supabase.table("awards").insert([{
                "candidate_id": cid,
                "award_type": clean_string(a.get("award_type")),
                "detail": clean_string(a.get("detail"))
            } for a in awards]).execute()

        # 6. References
        refs = parsed_json.get("references", [])
        if refs:
            supabase.table("references_table").insert([{
                "candidate_id": cid,
                "reference_name": clean_string(r.get("reference_name")),
                "designation": clean_string(r.get("designation")),
                "address": clean_string(r.get("address")),
                "phone": clean_string(r.get("phone")),
                "email": clean_string(r.get("email"))
            } for r in refs]).execute()

        # 7. Research Outputs
        res_out = parsed_json.get("research_outputs", [])
        if res_out:
            supabase.table("research_outputs").insert([{
                "candidate_id": cid,
                "title": clean_string(r.get("title")),
                "venue_name": clean_string(r.get("venue_name")),
                "output_type": validate_enum(r.get("output_type"), ['Journal', 'Conference', 'Book', 'Patent'], 'Unknown'),
                "publication_year": extract_int(r.get("publication_year")),
                "impact_factor": extract_float(r.get("impact_factor")),
                "author_names": clean_string(r.get("author_names")),
                "research_topics": r.get("research_topics", []) if isinstance(r.get("research_topics"), list) else []
            } for r in res_out]).execute()

        # 8. Supervision
        supv = parsed_json.get("supervision", [])
        if supv:
            supabase.table("supervision").insert([{
                "candidate_id": cid,
                "student_name": clean_string(s.get("student_name")),
                "degree_level": validate_enum(s.get("degree_level"), ['BS', 'MS', 'PhD'], 'Unknown'),
                "status": validate_enum(s.get("status"), ['Completed', 'Ongoing'], 'Unknown')
            } for s in supv]).execute()

        # 9. Skills
        skills = parsed_json.get("skills", [])
        if skills:
            supabase.table("skills").insert([{
                "candidate_id": cid,
                "skill_name": clean_string(sk.get("skill_name") or "Unknown Skill"),
                "skill_category": validate_enum(sk.get("skill_category"), ['Technical', 'Research', 'Soft', 'Tool'], 'Unknown')
            } for sk in skills]).execute()

        print(f"✅ Successfully saved: {clean_string(p_info.get('full_name')) or 'Unknown'}")
        return True
    except Exception as e:
        print(f"❌ DB Save Error: {e}")
        return False

# --- PIPELINE EXECUTION ---
def process_massive_pdf(pdf_path):
    print(f"\n🚀 Starting extraction from: {pdf_path}")
    try:
        text_parts = []
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                extracted = page.extract_text()
                if extracted: text_parts.append(extracted)
        raw_text = "\n".join(text_parts)
    except Exception as e:
        print(f"❌ Failed to read PDF: {e}")
        return

    cleaned_text = re.sub(r'\s+', ' ', raw_text).strip()
    raw_chunks = re.split(r'(?i)candidate for the post of', cleaned_text)
    candidate_chunks = ["Candidate for the post of " + c for c in raw_chunks if len(c.strip()) > 500]
    print(f"🧩 Successfully split document into {len(candidate_chunks)} candidates!\n")

    model = genai.GenerativeModel('gemini-3.1-flash-lite-preview') 
    
    for index, chunk_text in enumerate(candidate_chunks, 1):
        print(f"--- Processing Candidate {index} of {len(candidate_chunks)} ---")
        
        max_attempts = 4
        for attempt in range(max_attempts):
            try:
                response = model.generate_content(
                    f"{SYSTEM_PROMPT}\n\nCV TEXT:\n{chunk_text[:10000]}", 
                    generation_config={"response_mime_type": "application/json"}
                )
                parsed_data = json.loads(response.text)
                source_id = f"Handler (8).pdf - Candidate {index}"
                success = save_to_database(parsed_data, source_id)
                
                if success:
                    time.sleep(6) 
                    break 
                else:
                    raise ValueError("Database insertion failed due to schema mismatch.")
                
            except exceptions.ResourceExhausted:
                wait_time = 10 * (attempt + 1)
                print(f"⏳ Rate limit hit! Sleeping for {wait_time} seconds...")
                time.sleep(wait_time)
            except json.JSONDecodeError:
                print(f"⚠️ JSON parsing error on attempt {attempt + 1}. Retrying...")
                time.sleep(4)
            except Exception as e:
                print(f"⚠️ Unexpected error: {e}. Retrying...")
                time.sleep(4)
        else:
            print(f"❌ Skipping Candidate {index} after {max_attempts} failed attempts.")

# EXECUTE 
pdf_file_path = '/kaggle/input/datasets/asmrhi/talash-sample-cvs/Handler (8).pdf'
if os.path.exists(pdf_file_path):
    process_massive_pdf(pdf_file_path)
else:
    print(f"❌ Cannot find PDF at {pdf_file_path}. Please check the Kaggle path.")


🚀 Starting extraction from: /kaggle/input/datasets/asmrhi/talash-sample-cvs/Handler (8).pdf
🧩 Successfully split document into 43 candidates!

--- Processing Candidate 1 of 43 ---
✅ Successfully saved: MUHAMMAD SALMAN
--- Processing Candidate 2 of 43 ---
✅ Successfully saved: Aamina Akbar
--- Processing Candidate 3 of 43 ---
✅ Successfully saved: MUHAMMAD SHAHWAZ
--- Processing Candidate 4 of 43 ---
✅ Successfully saved: shaheer
--- Processing Candidate 5 of 43 ---
✅ Successfully saved: Nasir Ali Shah
--- Processing Candidate 6 of 43 ---
✅ Successfully saved: Muhammad Farrukh Qureshi
--- Processing Candidate 7 of 43 ---
✅ Successfully saved: Idris Khan
--- Processing Candidate 8 of 43 ---
✅ Successfully saved: MUHAMMAD MAJID
--- Processing Candidate 9 of 43 ---
✅ Successfully saved: Muhammad Sarmad Tariq
--- Processing Candidate 10 of 43 ---
✅ Successfully saved: Farman Ali
--- Processing Candidate 11 of 43 ---
✅ Successfully saved: Fiza Murtaza
--- Processing Candidate 12 of 43 ---
✅

In [4]:
# ==========================================
# CELL 4: MILESTONE EXPORT & DOWNLOAD
# ==========================================
def final_milestone_export():
    tables = [
        "candidates", "education", "experience", "certifications", 
        "awards", "references_table", "research_outputs", "supervision", "skills"
    ]
    print(f"\n🚀 Exporting full schema data from Supabase...")

    for table_name in tables:
        try:
            response = supabase.table(table_name).select("*").execute()
            if response.data:
                df = pd.DataFrame(response.data)
                filename = f"/kaggle/working/cv_exports/talash_{table_name}.csv"
                df.to_csv(filename, index=False)
                print(f"✅ {table_name.capitalize()}: {len(df)} rows exported.")
            else:
                print(f"⚠️ {table_name.capitalize()}: Table is empty.")
        except Exception as e:
            print(f"❌ Error exporting {table_name}: {e}")

final_milestone_export()

source_dir = '/kaggle/working/cv_exports'
zip_filename = '/kaggle/working/talash_complete_data' 
if os.path.exists(source_dir):
    print("\n📦 Zipping all 9 CSV files...")
    shutil.make_archive(zip_filename, 'zip', source_dir)
    print("✅ Zip file ready!")
    display(FileLink('talash_complete_data.zip'))
else:
    print(f"❌ Could not find the directory: {source_dir}")


🚀 Exporting full schema data from Supabase...
✅ Candidates: 43 rows exported.
✅ Education: 223 rows exported.
✅ Experience: 137 rows exported.
✅ Certifications: 26 rows exported.
✅ Awards: 80 rows exported.
✅ References_table: 86 rows exported.
✅ Research_outputs: 136 rows exported.
⚠️ Supervision: Table is empty.
✅ Skills: 62 rows exported.

📦 Zipping all 9 CSV files...
✅ Zip file ready!


/kaggle/working/talash_complete_data.zip